# Render the Intro Presenter talking clips with SadTalker (free GPU)

Runs SadTalker on a Colab T4 GPU to lip-sync `presenter.jpg` to the Magpie voice
mp3s, producing `seg-*.mp4` / `greet-*.mp4` + a patched `timeline.json`.

**First: Runtime → Change runtime type → GPU (T4).** Then run all cells.
At the end a zip downloads — unzip it into your repo's `public/intro/` and commit.
All 7 clips take ~2-3 min on GPU (vs hours on a CPU laptop).

In [ ]:
# 0. Confirm GPU is on
!nvidia-smi -L || print('NO GPU — set Runtime > Change runtime type > GPU')

In [ ]:
# 1. Clone the portfolio repo (has presenter.jpg + voice mp3s + the runner)
!git clone -b fix-issues https://github.com/vaibhavkumar07/portfolioV2.0.git
%cd portfolioV2.0

In [ ]:
# 2. Get SadTalker + model checkpoints (~2 GB, a couple of minutes)
!git clone --depth 1 https://github.com/OpenTalker/SadTalker tools/SadTalker
!cd tools/SadTalker && bash scripts/download_models.sh

In [ ]:
# 3. Install SadTalker deps (Colab already has torch + ffmpeg)
!pip install -q face_alignment==1.3.5 kornia yacs scikit-image librosa resampy \
    pydub av safetensors numba basicsr facexlib gfpgan imageio-ffmpeg

In [ ]:
# 4. Render all 7 clips on GPU (auto-patches SadTalker for numpy>=2 / new torchvision)
!python scripts/sadtalker_intro.py

In [ ]:
# 5. Zip the videos + patched manifest and download
import shutil, os
os.chdir('/content/portfolioV2.0')
!cd public/intro && zip -j /content/intro_videos.zip *.mp4 timeline.json
from google.colab import files
files.download('/content/intro_videos.zip')

## Back on your machine

1. Unzip `intro_videos.zip` into `public/intro/` (overwrites `timeline.json`,
   adds `greet-*.mp4` and `seg-*.mp4`).
2. `git add public/intro && git commit -m "Add SadTalker talking-head clips" && git push`.

The intro player auto-switches to the lip-synced video (the `video` fields are now
in `timeline.json`). The coded avatar / canvas portrait stays as the fallback.